In [ ]:
import pandas as pd
from classes.museum import Museum
from classes.specialty import Specialty
from classes.exhibit import Exhibit
from classes.curator import Curator
from classes.visitors import Visitors

f = 'data/g21_Museums_Exhibits_fv.csv'
db = 'data/database.db'

modelos = [Museum, Specialty, Exhibit, Curator, Visitors]
for c in modelos:
    c.path = db

try:
    df = pd.read_csv(f, sep=None, engine='python', encoding='utf-8-sig')
except:
    df = pd.read_excel(f)

df.columns = [str(c).strip().lower() for c in df.columns]

if 'visitors_numb' in df.columns:
    df['visitors_numb'] = pd.to_numeric(df['visitors_numb'], errors='coerce').fillna(0).astype(int)
else:
    df['visitors_numb'] = 0

df = df.fillna('')

for _, row in df[['specialty_id', 'curator_specialty']].drop_duplicates(subset=['specialty_id']).iterrows():
    if str(row['specialty_id']) != '':
        obj = Specialty(id=int(row['specialty_id']), specialty_name=row['curator_specialty'])
        Specialty.insert(obj.id)

for _, row in df[['museum_id', 'name']].drop_duplicates(subset=['museum_id']).iterrows():
    if str(row['museum_id']) != '':
        obj = Museum(id=int(row['museum_id']), name=row['name'])
        Museum.insert(obj.id)

for _, row in df[['exhibit_id', 'creation_date', 'title', 'category']].drop_duplicates(subset=['exhibit_id']).iterrows():
    if str(row['exhibit_id']) != '':
        dt = str(row['creation_date']).split(' ')[0] if row['creation_date'] else '2024-01-01'
        obj = Exhibit(id=int(row['exhibit_id']), creation_date=dt, title=row['title'], category=row['category'])
        Exhibit.insert(obj.id)

for _, row in df[['curator_id', 'extra_info', 'museum_id', 'specialty_id']].drop_duplicates(subset=['curator_id']).iterrows():
    if str(row['curator_id']) != '':
        obj = Curator(id=int(row['curator_id']), extra_info=row['extra_info'], id_museum=int(row['museum_id']), id_specialty=int(row['specialty_id']))
        Curator.insert(obj.id)

v_id = 1
for _, row in df[['exhibit_id', 'museum_id', 'end_date', 'visitors_numb']].drop_duplicates().iterrows():
    dt_v = str(row['end_date']).split(' ')[0] if row['end_date'] else '2024-01-01'
    obj = Visitors(id=v_id, id_exhibit=int(row['exhibit_id']), id_museum=int(row['museum_id']), conclusion_date=dt_v, n_visitors=int(row['visitors_numb']))
    Visitors.insert(obj.id)
    v_id += 1

print("Concluído")